# Method-consistent rerun of the Real rows of Table E.4 (B3, 2025)

**Purpose.** The Real rows of Table E.4 were produced on 24 May 2026 by a standalone driver whose GA arms differ from the Nominal rows'. This notebook reruns them with the **method-consistent** driver: the one `patch_Real_2025.py` generates from the deposited `B3_ablation_driver_v3.py` (GA arms `fit_eoga_beniwal_ga` / `fit_eiga_beniwal_ga`, Wang-faithful $\varepsilon^*$ nested in every fitness evaluation).

**Runtime (from the deposit's own logs):** e\*-IGA-SVR ≈ 2–4 h; e\*-OGA-SVR ≈ 4–12 h. **Not 1.6 h.**

**Plan — two sessions (the driver is resume-capable):**
| Session | `SESSION` | Runs | Where |
|:--|:--|:--|:--|
| 1 | `"IGA"` | e\*-IGA-SVR only | free Colab is enough |
| 2 | `"BOTH"` | e\*-OGA-SVR (IGA is skipped: already in the CSV) | Colab Pro+ (background execution) or your own PC |

**Needs:** `epsilon-IGA-SVR_EGX30_Full_Source_v2_09_169.zip` and the withheld `egx30_2000_2025.csv` (SHA-256 is checked below), both in Google Drive.

Keep the tab open and the computer awake on free Colab. Results are written to the VM's local disk and copied to Drive after the run.

In [ ]:
# 1. Settings -- edit these two lines
SESSION  = "IGA"      # "IGA" for session 1, "BOTH" for session 2
DRIVE_IN = "/content/drive/MyDrive/SVR_Profile/B3_Real_rerun"   # holds the zip and the price CSV

import os, sys, platform, hashlib, shutil, zipfile, subprocess, json, time, io
WORK = "/content/b3real"; os.makedirs(WORK, exist_ok=True)
OUT_LOCAL = "/content/Outputs_2025_B3_Real_consistent"
OUT_NAME  = "Outputs_2025_B3_Real_consistent"
ZIP_NAME, PRICE_NAME = "epsilon-IGA-SVR_EGX30_Full_Source_v2_09_169.zip", "egx30_2000_2025.csv"

# --- Access Google Drive -------------------------------------------------------------
# drive.mount() can fail with "MessageError: credential propagation was unsuccessful"
# (auth pop-up blocked/closed, third-party cookies blocked, or the browser is signed in
# to a different Google account than the Colab notebook). We therefore try, in order:
#   A. drive.mount (normal FUSE mount)
#   B. auth.authenticate_user + Drive API: the zip/CSV are downloaded into a local
#      mirror of DRIVE_IN, and the outputs are uploaded back to Drive in cell 6
#   C. manual upload of the two files from your computer (outputs zipped for download)
from google.colab import drive, auth
DRIVE_MODE = None
try:
    drive.mount('/content/drive', force_remount=True)
    DRIVE_MODE = "mount"
except Exception as e:   # MessageError: credential propagation was unsuccessful
    print("drive.mount failed:", e, "\n-> falling back to the Drive API")
    try: drive.flush_and_unmount()
    except Exception: pass

if DRIVE_MODE is None:
    try:
        auth.authenticate_user()
        from googleapiclient.discovery import build
        from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload
        _svc = build("drive", "v3", cache_discovery=False)
        _FOLDER = "application/vnd.google-apps.folder"
        def _child(parent, name, folder=False):
            q = "'%s' in parents and name = '%s' and trashed = false" % (parent, name.replace("'", "\\'"))
            if folder: q += " and mimeType = '%s'" % _FOLDER
            r = _svc.files().list(q=q, fields="files(id,name,mimeType)", spaces="drive").execute()["files"]
            return r[0]["id"] if r else None
        def _folder_id(path, create=False):
            fid = "root"
            for part in path.split("/content/drive/MyDrive/", 1)[1].strip("/").split("/"):
                nxt = _child(fid, part, folder=True)
                if nxt is None:
                    if not create: return None
                    nxt = _svc.files().create(body={"name": part, "mimeType": _FOLDER, "parents": [fid]},
                                              fields="id").execute()["id"]
                fid = nxt
            return fid
        def _download(file_id, dest):
            with io.FileIO(dest, "wb") as fh:
                dl = MediaIoBaseDownload(fh, _svc.files().get_media(fileId=file_id), chunksize=64 << 20)
                done = False
                while not done: _, done = dl.next_chunk()
        # local mirror of DRIVE_IN so cells 2 and 4 run unchanged
        _in_id = _folder_id(DRIVE_IN)
        assert _in_id, f"Drive folder not found: {DRIVE_IN}"
        DRIVE_IN_REMOTE, DRIVE_IN = DRIVE_IN, "/content/drive_in_mirror"
        os.makedirs(DRIVE_IN, exist_ok=True)
        for n in (ZIP_NAME, PRICE_NAME):
            fid = _child(_in_id, n); assert fid, f"{n} not found in {DRIVE_IN_REMOTE}"
            _download(fid, os.path.join(DRIVE_IN, n)); print("downloaded", n)
        _out_id = _child(_in_id, OUT_NAME, folder=True)
        if _out_id:   # session-1 results, so session 2 can resume
            os.makedirs(os.path.join(DRIVE_IN, OUT_NAME), exist_ok=True)
            for f in _svc.files().list(q="'%s' in parents and trashed = false" % _out_id,
                                       fields="files(id,name,mimeType)").execute()["files"]:
                if f["mimeType"] != _FOLDER:
                    _download(f["id"], os.path.join(DRIVE_IN, OUT_NAME, f["name"]))
        DRIVE_MODE = "api"
    except Exception as e:
        print("Drive API fallback failed:", repr(e), "\n-> falling back to manual upload")

if DRIVE_MODE is None:
    from google.colab import files
    DRIVE_IN = "/content/drive_in_mirror"; os.makedirs(DRIVE_IN, exist_ok=True)
    print(f"Upload {ZIP_NAME} and {PRICE_NAME} (and, for session 2, the session-1 output files):")
    for n, data in files.upload().items():
        dest = DRIVE_IN if n in (ZIP_NAME, PRICE_NAME) else os.path.join(DRIVE_IN, OUT_NAME)
        os.makedirs(dest, exist_ok=True); open(os.path.join(dest, n), "wb").write(data)
    for n in (ZIP_NAME, PRICE_NAME):
        assert os.path.exists(os.path.join(DRIVE_IN, n)), f"{n} was not uploaded"
    DRIVE_MODE = "upload"

OUT_DRIVE = os.path.join(DRIVE_IN, OUT_NAME)

def save_outputs_to_drive():
    """Called from cell 6 after OUT_LOCAL has been copied into OUT_DRIVE."""
    if DRIVE_MODE == "mount":
        drive.flush_and_unmount(); print("outputs flushed to", OUT_DRIVE)   # make sure writes reach Drive
    elif DRIVE_MODE == "api":
        out_id = _folder_id(os.path.join(DRIVE_IN_REMOTE, OUT_NAME), create=True)
        for n in sorted(os.listdir(OUT_DRIVE)):
            p = os.path.join(OUT_DRIVE, n)
            if not os.path.isfile(p): continue
            media, fid = MediaFileUpload(p, resumable=True), _child(out_id, n)
            if fid: _svc.files().update(fileId=fid, media_body=media).execute()
            else:   _svc.files().create(body={"name": n, "parents": [out_id]}, media_body=media).execute()
            print("uploaded", n)
    else:
        from google.colab import files
        z = shutil.make_archive(f"/content/{OUT_NAME}_{SESSION}", "zip", OUT_DRIVE)
        files.download(z)   # keep this zip: upload its contents again in session 2

print("Drive access mode:", DRIVE_MODE, "| inputs in", DRIVE_IN)
print(sys.version); print(platform.platform()); print("CPUs:", os.cpu_count())

In [ ]:
# 2. Unpack the archive and verify the withheld price file
ZIP = os.path.join(DRIVE_IN, ZIP_NAME)
PRICE = os.path.join(DRIVE_IN, "egx30_2000_2025.csv")
EXPECTED_SHA256 = "e35bebbeaf4d178f679e416b8dd2647882772a289b10c4fd5604149128b48b0c"  # Data/RETRIEVAL_MANIFEST.md
with zipfile.ZipFile(ZIP) as z:
    z.extractall("/content")
    tops = {n.split("/")[0] for n in z.namelist() if "/" in n}
A = "/content/epsilon-IGA-SVR_EGX30_Full_Source"
if not os.path.isdir(A) and len(tops) == 1: A = os.path.join("/content", tops.pop())  # top folder renamed in this release
assert os.path.isdir(os.path.join(A, "Data")), f"unexpected archive layout: {A}"
got = hashlib.sha256(open(PRICE, "rb").read()).hexdigest()
assert got == EXPECTED_SHA256, f"price file SHA-256 {got} does not match the manifest"
shutil.copy2(PRICE, os.path.join(A, "Data", "egx30_2000_2025.csv"))
print("price file verified; archive at", A)

In [ ]:
# 3. Rebuild the method-consistent Real driver exactly as on the VM (patch chain)
os.chdir(WORK)
os.makedirs("Scripts_v2_2", exist_ok=True); os.makedirs("Scripts_v2_2_B3", exist_ok=True)
src = open(os.path.join(A, "EGX30_IGA_SVR_v2_2_Colab.py"), encoding="utf-8").read()
# path rewrite applied on the VM in April-May 2026 (README section 2): username and Outputs -> Outputs_v2_2 only
src = src.replace(r"C:\Users\sabahhegazy\Documents\GARCH_SVR\Data", r"C:\Users\sabah_hhs_odesk_13\Documents\GARCH_SVR\Data")
src = src.replace(r"C:\Users\sabahhegazy\Documents\GARCH_SVR\Outputs", r"C:\Users\sabah_hhs_odesk_13\Documents\GARCH_SVR\Outputs_v2_2")
open("Scripts_v2_2/EGX30_IGA_SVR_v2_2_Colab.py", "w", encoding="utf-8").write(src)
for f in ["B3_ablation_driver_v3.py", "beniwal_ga_oga.py", "beniwal_ga_iga.py"]:
    shutil.copy2(os.path.join(A, f), os.path.join("Scripts_v2_2_B3", f))
for f in ["patch_v22_and_B3_2025.py", "patch_Real_2025.py"]:
    shutil.copy2(os.path.join(A, f), f)
for p in ["patch_v22_and_B3_2025.py", "patch_Real_2025.py"]:
    for stale in ["EGX30_IGA_SVR_v2_2_2025.py","B3_ablation_driver_2025.py",
                  "EGX30_IGA_SVR_v2_2_2025_Real.py","B3_ablation_driver_2025_Real.py"]:
        if p == "patch_v22_and_B3_2025.py" and os.path.exists(stale): os.remove(stale)
    r = subprocess.run([sys.executable, p], capture_output=True, text=True); print(r.stdout[-800:])
    assert r.returncode == 0, r.stderr
DRIVER = "B3_ablation_driver_2025_Real.py"
size = os.path.getsize(DRIVER)
assert size == 29925, f"driver is {size} bytes; expected 29,925 (LF) = 30,673 (CRLF), the patcher's output"
code_ = open(DRIVER, encoding="utf-8").read()
assert "fit_eoga_beniwal_ga" in code_ and "fit_eiga_beniwal_ga" in code_, "not the method-consistent driver"
print("method-consistent driver rebuilt:", DRIVER, size, "bytes")

In [ ]:
# 4. Session-specific model list (session 1 runs e*-IGA-SVR only; session 2 runs the full list,
#    and the driver's resume logic skips e*-IGA-SVR because it is already in the CSV)
run_file = DRIVER
if SESSION == "IGA":
    txt = open(DRIVER, encoding="utf-8").read()
    old = "ABLATION_MODELS = ['e*-OGA-SVR', 'e*-IGA-SVR']"
    assert txt.count(old) == 1
    open("B3_driver_session1_IGA.py", "w", encoding="utf-8").write(txt.replace(old, "ABLATION_MODELS = ['e*-IGA-SVR']"))
    run_file = "B3_driver_session1_IGA.py"
# restore session-1 results from Drive so session 2 resumes
os.makedirs(OUT_LOCAL, exist_ok=True)
if os.path.isdir(OUT_DRIVE):
    for f in os.listdir(OUT_DRIVE): shutil.copy2(os.path.join(OUT_DRIVE, f), OUT_LOCAL)
    print("restored:", os.listdir(OUT_LOCAL))
print("session", SESSION, "->", run_file)

In [ ]:
# 5. Run (streams the log; expect hours). Paths point into the archive, not the VM's Windows paths.
env = dict(os.environ,
           B3_DATA_DIR=os.path.join(A, "Data"),
           B3_V22_TABLES=os.path.join(A, "Outputs_2025_v22_Real", "Tables"),
           B3_OUTPUT_DIR=OUT_LOCAL)
t0 = time.time()
p = subprocess.Popen([sys.executable, "-u", run_file], cwd=WORK, env=env,
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout: print(line, end="")
p.wait(); print(f"\nexit {p.returncode} after {(time.time()-t0)/3600:.2f} h")

In [ ]:
# 6. Save: environment record, then copy the local outputs to Drive
import numpy, pandas, sklearn, scipy
json.dump({"session": SESSION, "python": sys.version, "platform": platform.platform(),
           "numpy": numpy.__version__, "pandas": pandas.__version__,
           "sklearn": sklearn.__version__, "scipy": scipy.__version__,
           "driver_bytes_LF": os.path.getsize(os.path.join(WORK, DRIVER)),
           "finished_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())},
          open(os.path.join(OUT_LOCAL, f"Colab_Environment_{SESSION}.json"), "w"), indent=2)
os.makedirs(OUT_DRIVE, exist_ok=True)
for f in os.listdir(OUT_LOCAL): shutil.copy2(os.path.join(OUT_LOCAL, f), OUT_DRIVE)
save_outputs_to_drive()
print(pandas.read_csv(os.path.join(OUT_LOCAL, "Table_B3_Beniwal_GA_vs_DE.csv")))